# 05 - KV cache dump analysis

End-to-end walk-through of `tq_bench.kv_analysis`:

1. Load baseline + quantized KV dumps produced by the C++ `llama-kv-dump` tool.
2. Per-layer K/V distribution statistics, split by vision vs text tokens.
3. Outlier channel detection.
4. Beta-distribution fit quality of the rotated coordinates (the TurboQuant theoretical assumption).
5. Quantization error vs baseline, compared to the paper's theoretical Lloyd-Max values.

If real dumps are not yet available this notebook falls back to a **synthetic dump** so the pipeline can still be exercised.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

from tq_bench.kv_analysis import (
    KVDump,
    KVDumpWriter,
    compute_per_layer_stats,
    outlier_ratio_vision_vs_text,
    compare_dumps,
    summarize_against_theoretical,
    vision_vs_text_rotation_analysis,
    generate_full_report,
    THEORETICAL_MSE_PER_COORD,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## 1. Locate dumps

Expected layout:

```
results/kv_dumps/
  baseline/
    meta.json
    K_layer_0.bin ... K_layer_27.bin
    V_layer_0.bin ... V_layer_27.bin
  tq-3/
    ...
```

If the C++ tool is not wired yet we synthesise a tiny mock dump so the rest of the notebook still runs.

In [ ]:
RESULTS_DIR = Path('../../results/kv_dumps').resolve()

def make_mock_dump(dump_dir: Path, *, noise: float = 0.0, run_name: str = 'mock') -> KVDump:
    """Fallback synthetic dump matching Qwen3-VL-2B layout."""
    rng = np.random.default_rng(0 if run_name == 'baseline' else 1)
    n_tokens, n_layers, n_kv_head, head_dim = 32, 4, 2, 128
    K, V = {}, {}
    for layer in range(n_layers):
        K[layer] = rng.standard_normal((n_tokens, n_kv_head, head_dim)).astype(np.float32)
        V[layer] = rng.standard_normal((n_tokens, n_kv_head, head_dim)).astype(np.float32)
        if noise:
            K[layer] = K[layer] + rng.standard_normal(K[layer].shape).astype(np.float32) * noise
            V[layer] = V[layer] + rng.standard_normal(V[layer].shape).astype(np.float32) * noise
    mask = np.zeros(n_tokens, dtype=bool)
    mask[: n_tokens // 2] = True
    KVDumpWriter(dump_dir).write(
        K=K, V=V, vision_token_mask=mask, run_name=run_name,
        extra_meta={'model_path': 'mock/Qwen3-VL-2B', 'cache_type_k': 'f16', 'cache_type_v': 'f16'}
    )
    return KVDump(dump_dir)

baseline_dir = RESULTS_DIR / 'baseline'
quant_runs = {
    'lcpp-kv-4': RESULTS_DIR / 'lcpp-kv-4',
    'tq-4':      RESULTS_DIR / 'tq-4',
    'tq-3':      RESULTS_DIR / 'tq-3',
    'tqp-4':     RESULTS_DIR / 'tqp-4',
}

if baseline_dir.exists():
    baseline = KVDump(baseline_dir)
    quant_dumps = {name: KVDump(p) for name, p in quant_runs.items() if p.exists()}
    print(f'Loaded real baseline from {baseline_dir}')
else:
    print('No real dumps found - using synthetic mock data')
    tmp = Path('.').resolve() / '_mock_kv_dumps'
    tmp.mkdir(exist_ok=True)
    baseline = make_mock_dump(tmp / 'baseline', run_name='baseline', noise=0.0)
    quant_dumps = {
        'noisy-lo': make_mock_dump(tmp / 'noisy-lo', run_name='noisy-lo', noise=0.05),
        'noisy-hi': make_mock_dump(tmp / 'noisy-hi', run_name='noisy-hi', noise=0.20),
    }

print(f'Baseline: {baseline}')
for name, d in quant_dumps.items():
    print(f'  {name}: {d}')

## 2. Per-layer K/V distribution (vision vs text)

In [ ]:
dist_df = compute_per_layer_stats(baseline, separate_vision_text=True)
display(dist_df.head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for tt, sub in dist_df.groupby('token_type'):
    axes[0].plot(sub['layer'], sub['k_norm_mean'], marker='o', label=f'K ({tt})')
    axes[0].plot(sub['layer'], sub['v_norm_mean'], marker='x', linestyle='--', label=f'V ({tt})')
    axes[1].plot(sub['layer'], sub['kv_norm_ratio'], marker='o', label=tt)

axes[0].set_title('Per-token L2 norm (mean)')
axes[0].set_xlabel('layer')
axes[0].set_ylabel('E[||row||]')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=7)

axes[1].set_title('K/V norm ratio per layer')
axes[1].axhline(1.0, color='gray', lw=1)
axes[1].set_xlabel('layer')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## 3. Outlier channels

Flags channels (head, dim) whose norm exceeds 10 x median.  The community claim is that 5-20% of K channels are 10-100 x outliers, and that this is what drives 2-bit quantization failure.

In [ ]:
outlier_df = outlier_ratio_vision_vs_text(baseline, threshold=10.0)
display(outlier_df.head(10))

if not outlier_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for tt, sub in outlier_df.groupby('token_type'):
        ax.plot(sub['layer'], sub['k_outlier_ratio'], marker='o', label=f'K ({tt})')
        ax.plot(sub['layer'], sub['v_outlier_ratio'], marker='x', linestyle='--', label=f'V ({tt})')
    ax.set_title('Outlier channel ratio (> 10x median) per layer')
    ax.set_xlabel('layer')
    ax.set_ylabel('outlier fraction')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 4. TurboQuant rotation diagnostics

For every layer, apply FWHT + deterministic sign flip (seed=42) to each KV vector, then test whether the rotated coordinates look `Beta((d-1)/2, (d-1)/2)`.

**Research question:** does the Beta assumption hold equally well for vision vs text tokens?  If not, vision tokens pay extra quantization cost even at identical bitwidths.

In [ ]:
rot_df = vision_vs_text_rotation_analysis(baseline, kind='both')
display(rot_df.head(10))

if not rot_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for (tt, knd), sub in rot_df.groupby(['token_type', 'kind']):
        label = f'{knd} ({tt})'
        axes[0].plot(sub['layer'], sub['ks_statistic'], marker='o', label=label)
        axes[1].plot(sub['layer'], sub['mean_abs_correlation'], marker='o', label=label)
    axes[0].set_title('Beta fit KS statistic per layer (lower = better)')
    axes[0].set_xlabel('layer')
    axes[0].set_ylabel('KS statistic')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=7)
    axes[1].set_title('Rotated coord mean |correlation|')
    axes[1].set_xlabel('layer')
    axes[1].set_ylabel('mean |corr|')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    summary = rot_df.groupby(['token_type', 'kind']).agg(
        mean_ks=('ks_statistic', 'mean'),
        mean_p=('p_value', 'mean'),
        mean_corr=('mean_abs_correlation', 'mean'),
    )
    print('Aggregate Beta fit:')
    display(summary.round(4))

## 5. Quantization error vs theoretical

For each quantized run, compute per-layer diff metrics against the baseline and compare the measured per-coord MSE to the paper's Lloyd-Max prediction.

| bitwidth | theoretical MSE / coord |
| --- | --- |
| 2 | 0.117 |
| 3 | 0.034 |
| 4 | 0.009 |

In [ ]:
print('Theoretical (paper) values:', THEORETICAL_MSE_PER_COORD)

frames = []
for name, dump in quant_dumps.items():
    df = compare_dumps(baseline, dump)
    df.insert(0, 'run', name)
    frames.append(df)
quant_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if not quant_df.empty:
    summary = quant_df[quant_df['token_type'] == 'all'].groupby(['run', 'kind']).agg(
        mse=('per_coord_mse', 'mean'),
        cos=('cosine_sim', 'mean'),
        rel=('relative_error', 'mean'),
        ip_bias=('inner_product_bias', 'mean'),
    )
    display(summary.round(5))

    fig, ax = plt.subplots(figsize=(8, 4))
    for (run, kind), sub in quant_df[quant_df['token_type'] == 'all'].groupby(['run', 'kind']):
        sub = sub.sort_values('layer')
        ax.plot(sub['layer'], sub['per_coord_mse'], marker='o', label=f'{run} / {kind}')
    ax.set_yscale('log')
    ax.set_title('Per-coord MSE vs baseline (log scale)')
    ax.set_xlabel('layer')
    ax.set_ylabel('MSE / coord')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

## 6. Write full report

Dump everything to `results/kv_analysis/<timestamp>/` so downstream analysis can reuse the CSVs.

In [ ]:
import datetime as dt

out_dir = Path('../../results/kv_analysis') / dt.datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir.mkdir(parents=True, exist_ok=True)

artifacts = generate_full_report(
    baseline_dump=baseline,
    quant_dumps=quant_dumps,
    output_dir=out_dir,
    make_plots=True,
    bits_by_run={
        'tq-4': 4, 'tq-3': 3, 'tq-2': 2,
        'lcpp-kv-4': 4, 'lcpp-kv-8': 8,
        'tqp-4': 4, 'tqp-3': 3, 'tqp-5': 5,
    },
)
print('Wrote:')
for k, v in artifacts.items():
    print(f'  {k:35s} -> {v}')